# Проверки `c_nazn` в `ods.scd1_z_main_docum`

Цели:
1. Посмотреть все уникальные варианты `c_nazn` за первую неделю июня.
2. Выделить варианты `c_nazn`, которые могут относиться к эквайрингу.

Период задается параметрами ниже (по умолчанию: с `2026-06-01` по `2026-06-07` включительно).

In [ ]:
import re

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect


def normalize_text_value(raw_value):
    text = '' if raw_value is None else str(raw_value)
    text = text.replace('\u00A0', ' ').replace('\u202F', ' ').replace('\u2007', ' ')
    text = text.lower().replace('ё', 'е')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def cut_tail_for_semantics(normalized_text, contract_tail_cut_regex=r'(?i)(\bпо\s+договору\b).*$'):
    text = normalized_text or ''
    # Оставляем только смысловой префикс, отрезая хвост с реквизитами договора.
    text = re.sub(contract_tail_cut_regex, r'\1', text).strip()
    text = re.sub(r'\s+', ' ', text).strip(' ,;:-')
    return text


def make_semantic_key(text_value):
    return re.sub(r'[^0-9a-zа-я]+', '', text_value or '')


print('Imports and text helpers loaded')

In [ ]:
# Параметры периода: первая неделя июня
week_start = '2026-06-01'
week_end_exclusive = '2026-06-08'  # c 01 по 07 июня включительно

# Таблица-источник
table_name = 'ods.scd1_z_main_docum'

# Параметры подключения к Impala (как в 01_07_acq_dash.ipynb)
impala_db = 'sandbox_ai'
impala_queue = 'ai'
impala_user_name = 'Shestopalov-VYur'
impala_keytab_path = '/home/jovyan/test_requests/tech.keytab'
impala_use_credentials = True
impala_update_keytab = True

# Параметры выполнения
mem_limit = '8g'

# Правила семантической унификации
apply_contract_tail_cut = True
contract_tail_cut_regex = r'(?i)(\bпо\s+договору\b).*$'
use_semantic_key = True  # ключ без пробелов/знаков препинания

# Ловим слова от корня "эквайр" (эквайринг, эквайринга, эквайринговый и т.д.)
ekv_pattern_py = r'(?:^|[^а-яa-z0-9])эквайр[а-я]*(?:[^а-яa-z0-9]|$)'

# Параметры отображения таблиц
show_full_tables = False
preview_limit = 200

if show_full_tables:
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
else:
    pd.set_option('display.max_rows', preview_limit)
    pd.set_option('display.max_columns', 50)
    pd.set_option('display.max_colwidth', 140)

pd.set_option('display.width', 0)

print(f'Period: [{week_start}, {week_end_exclusive})')
print(f'table={table_name}')
print(
    f'show_full_tables={show_full_tables}, preview_limit={preview_limit}, '
    f'apply_contract_tail_cut={apply_contract_tail_cut}, use_semantic_key={use_semantic_key}'
)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': impala_queue},
    kerberos={
        'keytab_path': impala_keytab_path,
        'use_credentials': impala_use_credentials,
        'update_keytab': impala_update_keytab,
    },
    user_params={'user_name': impala_user_name},
)
imp._init_connection()
print('Impala connection initialized')

## 1) Один SQL в Impala: выгружаем все варианты `c_nazn`

In [ ]:
sql_base_variants = f"""
select
    coalesce(c_nazn, '') as c_nazn_raw,
    count(*) as raw_cnt
from {table_name}
where cast(c_date_prov as date) >= date '{week_start}'
  and cast(c_date_prov as date) < date '{week_end_exclusive}'
group by coalesce(c_nazn, '')
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    base_variants_df = imp.fetch(sql_base_variants)

if 'c_nazn_raw' not in base_variants_df.columns or 'raw_cnt' not in base_variants_df.columns:
    raise RuntimeError('base_variants_df must contain columns: c_nazn_raw, raw_cnt')

base_variants_df['c_nazn_raw'] = base_variants_df['c_nazn_raw'].map(lambda x: '' if x is None else str(x))
base_variants_df['raw_cnt'] = pd.to_numeric(base_variants_df['raw_cnt'], errors='coerce').fillna(0).astype('int64')

print(f'Rows in base_variants_df: {len(base_variants_df):,}')
print(f'Total scoped rows by count: {int(base_variants_df["raw_cnt"].sum()):,}')

base_variants_preview_df = base_variants_df.sort_values(
    ['raw_cnt', 'c_nazn_raw'], ascending=[False, True], kind='stable'
)

if show_full_tables:
    display(base_variants_preview_df)
else:
    display(base_variants_preview_df.head(preview_limit))

In [ ]:
if 'base_variants_df' not in globals():
    raise RuntimeError('Run base variants load cell first')

# Линейная обработка только в pandas
work_variants_df = base_variants_df.copy()
work_variants_df['c_nazn_norm'] = work_variants_df['c_nazn_raw'].map(normalize_text_value)

if apply_contract_tail_cut:
    work_variants_df['c_nazn_semantic'] = work_variants_df['c_nazn_norm'].map(
        lambda value: cut_tail_for_semantics(value, contract_tail_cut_regex)
    )
else:
    work_variants_df['c_nazn_semantic'] = work_variants_df['c_nazn_norm']

if use_semantic_key:
    work_variants_df['c_nazn_key'] = work_variants_df['c_nazn_semantic'].map(make_semantic_key)
else:
    work_variants_df['c_nazn_key'] = work_variants_df['c_nazn_semantic']

work_variants_df['is_ekv'] = work_variants_df['c_nazn_norm'].str.contains(
    ekv_pattern_py,
    regex=True,
    na=False,
)

work_variants_df = work_variants_df.sort_values(
    ['raw_cnt', 'c_nazn_semantic'], ascending=[False, True], kind='stable'
)

unique_stats_df = pd.DataFrame([
    {
        'total_rows': int(work_variants_df['raw_cnt'].sum()),
        'unique_raw_c_nazn_count': int(work_variants_df['c_nazn_raw'].nunique(dropna=False)),
        'unique_semantic_count': int(work_variants_df['c_nazn_semantic'].nunique(dropna=False)),
        'unique_key_count': int(work_variants_df['c_nazn_key'].nunique(dropna=False)),
    }
])

unique_variants_df = (
    work_variants_df.groupby('c_nazn_key', dropna=False, as_index=False)
    .agg(
        c_nazn=('c_nazn_semantic', 'first'),
        cnt=('raw_cnt', 'sum'),
        raw_variants_collapsed=('c_nazn_raw', 'nunique'),
    )
    .sort_values(['cnt', 'c_nazn'], ascending=[False, True], kind='stable')
    .reset_index(drop=True)
)

display(unique_stats_df)
print(f'Rows in unique_variants_df: {len(unique_variants_df):,}')

if show_full_tables:
    display(unique_variants_df)
else:
    display(unique_variants_df.head(preview_limit))

## 2) Варианты `c_nazn` по эквайрингу (из уже загруженного `base_variants_df`)

In [ ]:
if 'work_variants_df' not in globals():
    raise RuntimeError('Run unique variants cell first')

ekv_scope_df = work_variants_df[work_variants_df['is_ekv']].copy()

ekv_stats_df = pd.DataFrame([
    {
        'scoped_rows': int(work_variants_df['raw_cnt'].sum()),
        'matched_rows': int(ekv_scope_df['raw_cnt'].sum()),
        'matched_unique_semantic': int(ekv_scope_df['c_nazn_semantic'].nunique(dropna=False)),
        'matched_unique_key': int(ekv_scope_df['c_nazn_key'].nunique(dropna=False)),
    }
])

display(ekv_stats_df)

In [ ]:
if 'work_variants_df' not in globals():
    raise RuntimeError('Run unique variants cell first')

ekv_scope_df = work_variants_df[work_variants_df['is_ekv']].copy()
ekv_scope_df = ekv_scope_df.sort_values(['raw_cnt', 'c_nazn_semantic'], ascending=[False, True], kind='stable')

ekv_variants_df = (
    ekv_scope_df.groupby('c_nazn_key', dropna=False, as_index=False)
    .agg(
        c_nazn=('c_nazn_semantic', 'first'),
        cnt=('raw_cnt', 'sum'),
        raw_variants_collapsed=('c_nazn_raw', 'nunique'),
    )
    .sort_values(['cnt', 'c_nazn'], ascending=[False, True], kind='stable')
    .reset_index(drop=True)
)

print(f'Rows in ekv_variants_df: {len(ekv_variants_df):,}')

if show_full_tables:
    display(ekv_variants_df)
else:
    display(ekv_variants_df.head(preview_limit))

In [ ]:
# Необязательно: сохранить результаты в CSV
save_to_csv = False
unique_out_path = './c_nazn_unique_first_week_june.csv'
ekv_out_path = './c_nazn_ekv_first_week_june.csv'

if save_to_csv:
    unique_variants_df.to_csv(unique_out_path, index=False)
    ekv_variants_df.to_csv(ekv_out_path, index=False)
    print(f'Saved: {unique_out_path}')
    print(f'Saved: {ekv_out_path}')
else:
    print('save_to_csv=False, nothing was written.')